# LeafDoc — Plant Disease Model Training (Google Colab)

This notebook trains the **MobileNetV3Large** plant disease classifier AND a **leaf-vs-not-leaf** binary gate that is used at inference time to reject obviously out-of-scope images (anything that isn't a plant leaf).

**What you get at the end:**
- `plant_disease_model.keras` — 38-class disease classifier
- `class_indices.json` — ordered list of class names
- `leaf_classifier.keras` — binary leaf vs non-leaf model
- A suggested `LEAF_THRESHOLD` value for your backend `.env`

**Before you run this notebook:**
1. Click `Runtime → Change runtime type → Hardware accelerator: GPU (T4)`.
2. Have a Kaggle API token ready (see `colab/README.md` for how to get `kaggle.json`).
3. Be signed in to the Google account whose Drive you want artifacts saved to.

**Estimated runtime:** ~45–75 min on a T4 (depends on Colab load).

## 1. Environment check

In [ ]:
!nvidia-smi || echo 'No GPU detected. Go to Runtime > Change runtime type > GPU.'
import sys, platform
print('Python:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Install / verify dependencies. Colab already has tensorflow + most of these.
!pip install -q kaggle tensorflow-datasets

import tensorflow as tf
import keras
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

# IMPORTANT: copy the TensorFlow version printed above into
# leafdoc-backend/requirements.txt (replace the placeholder) so your local
# FastAPI server can load the .keras files this notebook produces.
print('\n>>> COPY THIS INTO leafdoc-backend/requirements.txt:')
print(f'tensorflow=={tf.__version__}')

## 2. Mount Google Drive

Your trained models and class indices will be saved to `MyDrive/leafdoc/` so they survive Colab session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/leafdoc'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Artifacts will be saved to:', DRIVE_DIR)

## 3. Kaggle credentials

1. Go to https://www.kaggle.com/settings → "Create New API Token". This downloads `kaggle.json`.
2. Run the cell below and upload that file when prompted.

In [ ]:
from google.colab import files
import os, shutil

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('kaggle.json installed.')
else:
    print('kaggle.json already present.')

## 4. Download the PlantVillage ("New Plant Diseases") dataset

In [ ]:
import os
DATA_ROOT = '/content/data'
os.makedirs(DATA_ROOT, exist_ok=True)

if not os.path.exists(f'{DATA_ROOT}/new-plant-diseases-dataset.zip'):
    !cd {DATA_ROOT} && kaggle datasets download -d vipoooool/new-plant-diseases-dataset

if not os.path.isdir(f'{DATA_ROOT}/New Plant Diseases Dataset(Augmented)'):
    !cd {DATA_ROOT} && unzip -q new-plant-diseases-dataset.zip

!ls {DATA_ROOT}

In [ ]:
# Path config — matches what leafdoc-backend/train_model.py expects
DATA_DIR = f'{DATA_ROOT}/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR = f'{DATA_DIR}/train'
VALID_DIR = f'{DATA_DIR}/valid'

print('Train:', TRAIN_DIR, '->', os.path.isdir(TRAIN_DIR))
print('Valid:', VALID_DIR, '->', os.path.isdir(VALID_DIR))
print('Number of classes:', len(os.listdir(TRAIN_DIR)))

## 5. Build the disease classifier model

MobileNetV3Large with `mobilenet_v3.preprocess_input` baked into the graph (so the FastAPI backend can pass raw [0, 255] arrays at inference time).

Augmentation (random flip + rotation) is applied in the `tf.data` pipeline above — NOT inside the model. This avoids a TF 2.20 / Keras 3.13 bug with `RandomFlip`/`RandomRotation` on symbolic inputs and also lets augmentation run on the CPU in parallel with GPU training.

In [ ]:
import tensorflow as tf
import json

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
LEARNING_RATE = 1e-3
EPOCHS_FROZEN = 10
EPOCHS_FINETUNE = 5
FINETUNE_LR = 1e-5
FINETUNE_UNFREEZE_FROM = -30  # unfreeze last 30 layers

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, seed=123, image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print(f'Found {len(class_names)} classes')

# Persist class indices immediately
CLASS_INDICES_PATH = f'{DRIVE_DIR}/class_indices.json'
with open(CLASS_INDICES_PATH, 'w') as f:
    json.dump(class_names, f, indent=2)
print('Saved class indices to', CLASS_INDICES_PATH)

AUTOTUNE = tf.data.AUTOTUNE

# Data augmentation lives in the tf.data pipeline (NOT inside the model).
# Reasons:
#   - sidesteps a TF 2.20 / Keras 3.13 bug where RandomFlip/RandomRotation
#     crash with `module 'keras' has no attribute 'KerasTensor'` when called
#     on symbolic inputs in the functional API
#   - augmentation runs on CPU in parallel with GPU training (faster)
#   - keeps the saved .keras file simple, so the FastAPI backend can pass raw
#     [0, 255] arrays at inference time without random flips ever happening
data_aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
], name='augmentation')

# IMPORTANT — memory-conscious pipeline for free Colab (~12 GB host RAM):
#   * No `.shuffle(N)` here. With 70k images at 224x224x3 float32, even a
#     1000-batch buffer would try to keep ~19 GB resident and trigger the
#     'could not allocate pinned host of size: 8 GiB' OOM kernel restart.
#     image_dataset_from_directory already shuffles file paths each epoch,
#     which is sufficient randomization for transfer learning.
#   * `.prefetch(buffer_size=2)` is a small literal instead of AUTOTUNE so XLA
#     can't auto-balloon prefetch buffers into multi-GiB pinned host blocks.
#   * `val_ds` is NOT cached: 17k float32 images would alone consume ~10 GB.
# If you still OOM, drop BATCH_SIZE from 32 to 16 above.
train_ds_p = (
    train_ds
        .map(lambda x, y: (data_aug(x, training=True), y), num_parallel_calls=AUTOTUNE)
        .prefetch(buffer_size=2)
)
val_ds_p = val_ds.prefetch(buffer_size=2)

In [ ]:
def build_disease_model(num_classes):
    base_model = tf.keras.applications.MobileNetV3Large(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    # Note: augmentation lives in the tf.data pipeline (see previous cell), NOT here.
    # Preprocessing IS baked into the model so the FastAPI backend can pass raw [0,255] arrays.
    x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs), base_model

model, base_model = build_disease_model(len(class_names))
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)
model.summary()

## 6. Stage 1 — train with frozen base (10 epochs)

In [ ]:
MODEL_PATH = f'{DRIVE_DIR}/plant_disease_model.keras'

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    MODEL_PATH, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)

history_frozen = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=EPOCHS_FROZEN,
    callbacks=[checkpoint_cb]
)

## 7. Stage 2 — fine-tune last 30 layers (5 epochs, low LR)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINETUNE_UNFREEZE_FROM]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(FINETUNE_LR),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

history_ft = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=EPOCHS_FINETUNE,
    callbacks=[checkpoint_cb]
)

## 8. Evaluate the disease classifier

In [ ]:
import numpy as np

loss, acc = model.evaluate(val_ds_p, verbose=1)
print(f'\nFinal validation accuracy: {acc * 100:.2f}%')

# Top-3 accuracy
top3 = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3)
for batch_x, batch_y in val_ds_p:
    preds = model(batch_x, training=False)
    top3.update_state(batch_y, preds)
print(f'Validation top-3 accuracy: {top3.result().numpy() * 100:.2f}%')

## 9. Build & train the leaf-vs-not-leaf gate

We train a small MobileNetV3Small binary classifier: **leaf** (any image from the PlantVillage train set) vs **not-leaf** (Imagenette images upscaled to 224x224). At inference time the FastAPI backend runs this gate first; if `leaf_prob < LEAF_THRESHOLD`, the image is rejected as 'not a leaf' before the disease classifier even sees it.

Imagenette is fast.ai's curated 10-class subset of ImageNet (fish, dog, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute) — none of which overlap with leaves, making them clean negatives. It's served from fast.ai's AWS S3 bucket, which is much more reliable than the Toronto CIFAR mirror (currently returning 503s).

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import random

LEAF_BATCH = 32
LEAF_TARGET_PER_CLASS = 4000  # ~4k positives, ~4k negatives

# Positives: random images from PlantVillage train set (label=1)
leaf_files = []
for cls in os.listdir(TRAIN_DIR):
    cls_dir = os.path.join(TRAIN_DIR, cls)
    if os.path.isdir(cls_dir):
        leaf_files.extend([os.path.join(cls_dir, f) for f in os.listdir(cls_dir)])
random.seed(42)
random.shuffle(leaf_files)
leaf_files = leaf_files[:LEAF_TARGET_PER_CLASS]
print(f'Sampled {len(leaf_files)} leaf images')

def load_leaf_ds(paths, label):
    def _load(p):
        img = tf.io.read_file(p)
        img = tf.io.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        return img, label
    ds = tf.data.Dataset.from_tensor_slices(paths)
    return ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)

pos_ds = load_leaf_ds(leaf_files, 1)

# Negatives: Imagenette via tensorflow-datasets.
# Why not CIFAR (10 or 100)?
#   - Both CIFAR variants are hosted on www.cs.toronto.edu which has been
#     returning 503 errors today. Imagenette is on s3.amazonaws.com (fast.ai's
#     bucket) — different infrastructure entirely, never observed flaky.
#   - Imagenette's 10 classes (fish, dog, cassette player, chain saw, church,
#     French horn, garbage truck, gas pump, golf ball, parachute) have ZERO
#     overlap with leaves -> clean negative samples.
#   - 160px native resolution is closer to our 224, so less upscaling artifact
#     than CIFAR's 32x32 — sharper leaf-classifier with better generalization.
neg_raw = tfds.load('imagenette/160px-v2', split='train', as_supervised=True)
def _neg_map(img, _label):
    img = tf.image.resize(img, IMG_SIZE)
    return img, 0
neg_ds = neg_raw.take(LEAF_TARGET_PER_CLASS).map(
    _neg_map, num_parallel_calls=tf.data.AUTOTUNE
)

leaf_ds = pos_ds.concatenate(neg_ds).shuffle(2000).batch(LEAF_BATCH).prefetch(tf.data.AUTOTUNE)

# Held-out validation: small fresh sample of each
pos_val_files = leaf_files[:500]  # reuse first 500 (intentional overlap is fine for a sanity check; for serious eval use a held-out split)
pos_val = load_leaf_ds(pos_val_files, 1).batch(LEAF_BATCH)

neg_val = (
    tfds.load('imagenette/160px-v2', split='validation', as_supervised=True)
        .take(500)
        .map(_neg_map)
        .batch(LEAF_BATCH)
)

In [ ]:
def build_leaf_classifier():
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet'
    )
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    return tf.keras.Model(inputs, out)

leaf_model = build_leaf_classifier()
leaf_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
leaf_model.fit(leaf_ds, epochs=5)

LEAF_MODEL_PATH = f'{DRIVE_DIR}/leaf_classifier.keras'
leaf_model.save(LEAF_MODEL_PATH)
print('Saved leaf classifier to', LEAF_MODEL_PATH)

## 10. Calibrate `LEAF_THRESHOLD`

We collect the leaf-classifier scores on validation leaves and on validation non-leaves, then suggest a threshold that maximizes balanced accuracy. Copy the suggested value into your backend `.env`.

In [ ]:
import numpy as np

pos_scores, neg_scores = [], []
for x, _ in pos_val:
    pos_scores.extend(leaf_model.predict(x, verbose=0).flatten().tolist())
for x, _ in neg_val:
    neg_scores.extend(leaf_model.predict(x, verbose=0).flatten().tolist())

pos_scores = np.array(pos_scores)
neg_scores = np.array(neg_scores)
print(f'Leaf scores  — mean: {pos_scores.mean():.3f}, min: {pos_scores.min():.3f}')
print(f'Non-leaf     — mean: {neg_scores.mean():.3f}, max: {neg_scores.max():.3f}')

# Sweep thresholds
best_t, best_bal = 0.5, 0
for t in np.arange(0.05, 0.96, 0.05):
    tpr = (pos_scores >= t).mean()
    tnr = (neg_scores < t).mean()
    bal = (tpr + tnr) / 2
    if bal > best_bal:
        best_bal, best_t = bal, t

print(f'\nSuggested LEAF_THRESHOLD={best_t:.2f}  (balanced accuracy {best_bal*100:.1f}%)')
print('\n>>> Copy this into your leafdoc-backend/.env file:')
print(f'LEAF_THRESHOLD={best_t:.2f}')

## 11. Sanity inference

Load the saved disease model and run it on a single validation image to make sure the saved file works.

In [ ]:
import numpy as np
from PIL import Image

loaded = tf.keras.models.load_model(MODEL_PATH)

# Pick first image of first class
first_class = sorted(os.listdir(VALID_DIR))[0]
first_image = sorted(os.listdir(os.path.join(VALID_DIR, first_class)))[0]
img_path = os.path.join(VALID_DIR, first_class, first_image)

img = Image.open(img_path).resize(IMG_SIZE)
arr = np.expand_dims(np.array(img, dtype=np.float32), 0)
probs = loaded.predict(arr)[0]
top3 = probs.argsort()[-3:][::-1]

print(f'Image: {img_path}')
print(f'True class: {first_class}')
print('Top 3 predictions:')
for i in top3:
    print(f'  {class_names[i]}: {probs[i]*100:.2f}%')

## 12. Download artifacts to your local machine

Run these cells to download the three files. Place them in `leafdoc-backend/models/`:
- `plant_disease_model.keras`
- `class_indices.json`
- `leaf_classifier.keras`

In [ ]:
from google.colab import files

files.download(MODEL_PATH)
files.download(CLASS_INDICES_PATH)
files.download(LEAF_MODEL_PATH)

print('\nAll three artifacts have also been saved to your Google Drive at /content/drive/MyDrive/leafdoc/.')